# Medicare Part D Spending and Utilization EDA (2016–2024)

This notebook consolidates all nine annual Medicare Part D workbooks into one analysis. Each workbook contains multiple calendar-year blocks; the analysis identifies the main spending sheet, profiles its fields, and compares the latest year reported in each workbook (2016–2024). It also summarizes workbook coverage and examines manufacturer and drug-use sheets.

All results are descriptive. The annual workbooks can overlap in their historical years, so the cross-workbook trend uses the latest year from each report to avoid double-counting those repeated years.

In [ ]:
from pathlib import Path
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "Datasets").exists():
    ROOT = ROOT.parent
MEDICARE_DIR = ROOT / "Datasets" / "Medicare Part D Spending by Drug-Excel Reports including Historical Data RY26"
FILES = sorted(MEDICARE_DIR.glob("Medicare Part D Spending by Drug DYT*/*.xlsx"))
print(f"Workbooks found: {len(FILES)}")
for path in FILES:
    print(f"- {path.parent.name}: {path.name} ({path.stat().st_size / 1024**2:.1f} MB)")


## Load and identify the main spending sheet

In [ ]:
def load_report(path):
    excel = pd.ExcelFile(path)
    main_sheet = next((s for s in excel.sheet_names if "spending" in s.lower() or "utilization" in s.lower()), excel.sheet_names[0])
    raw = pd.read_excel(path, sheet_name=main_sheet, header=None)
    header_row = next((i for i in range(min(15, len(raw))) if raw.iloc[i].astype(str).str.contains("Brand Name", case=False, na=False).any()), None)
    if header_row is None:
        raise ValueError(f"Could not find the Brand Name header in {path.name}")
    df = pd.read_excel(path, sheet_name=main_sheet, header=header_row)
    df.columns = [str(col).replace("\n", " ").strip() for col in df.columns]
    df = df.dropna(how="all").copy()
    report_year_match = re.search(r"DYT(\d{4})", path.parent.name)
    report_year = int(report_year_match.group(1)) if report_year_match else None
    spend_columns = [col for col in df.columns if str(col).lower().startswith("total spending")]
    if report_year is not None and spend_columns:
        latest_spend_column = spend_columns[-1]
        df[latest_spend_column] = pd.to_numeric(df[latest_spend_column], errors="coerce")
    else:
        latest_spend_column = None
    return df, {"sheets": excel.sheet_names, "main_sheet": main_sheet, "header_row": header_row,
                "report_year": report_year, "spend_columns": spend_columns, "latest_spend_column": latest_spend_column}

reports = []
for path in FILES:
    df, info = load_report(path)
    reports.append({"path": path, "data": df, "info": info})
    print(f"{info['report_year']}: {df.shape[0]:,} rows × {df.shape[1]:,} columns | {info['main_sheet']} | {len(info['spend_columns'])} spending-year columns")


## Workbook coverage and data quality

In [ ]:
coverage = []
for report in reports:
    df, info = report["data"], report["info"]
    coverage.append({
        "Report year": info["report_year"],
        "Main sheet": info["main_sheet"],
        "Rows": len(df),
        "Columns": len(df.columns),
        "Spending-year blocks": len(info["spend_columns"]),
        "Worksheets": ", ".join(info["sheets"]),
        "Duplicate rows": int(df.duplicated().sum()),
        "Missing cells (%)": round(df.isna().mean().mean() * 100, 2),
    })
coverage_df = pd.DataFrame(coverage).sort_values("Report year")
display(coverage_df)

for report in reports:
    df, info = report["data"], report["info"]
    print(f"\n### {info['report_year']} report — field profile")
    profile = pd.DataFrame({
        "Data type": df.dtypes.astype(str),
        "Missing count": df.isna().sum(),
        "Missing (%)": (df.isna().mean() * 100).round(2),
        "Distinct values": df.nunique(dropna=True),
    }).sort_values("Missing (%)", ascending=False)
    display(profile)


## Latest-year spending comparison (2016–2024)

In [ ]:
annual = []
latest_year_drugs = []
for report in reports:
    df, info = report["data"], report["info"]
    col = info["latest_spend_column"]
    if col is None:
        continue
    spend = pd.to_numeric(df[col], errors="coerce")
    annual.append({
        "Year": info["report_year"],
        "Total spending": spend.sum(),
        "Median drug spending": spend.median(),
        "Drugs with reported spending": int(spend.notna().sum()),
        "Drug rows": len(df),
    })
    brand = next((c for c in df.columns if "brand name" in c.lower()), None)
    generic = next((c for c in df.columns if "generic name" in c.lower()), None)
    if brand:
        cols = [brand, col] + ([generic] if generic else [])
        top = df[cols].copy().rename(columns={brand: "Brand name", col: "Spending", generic: "Generic name"} if generic else {brand: "Brand name", col: "Spending"})
        top["Spending"] = pd.to_numeric(top["Spending"], errors="coerce")
        top["Report year"] = info["report_year"]
        latest_year_drugs.append(top.nlargest(10, "Spending"))

annual_df = pd.DataFrame(annual).sort_values("Year").set_index("Year")
display(annual_df.style.format({"Total spending": "${:,.0f}", "Median drug spending": "${:,.0f}"}))
fig, ax = plt.subplots(figsize=(11, 5))
annual_df["Total spending"].plot(marker="o", ax=ax, color="#167d8d")
ax.set_title("Medicare Part D total drug spending by report year")
ax.set_ylabel("Total spending (USD)")
ax.set_xlabel("Report year")
ax.grid(alpha=.25)
fig.tight_layout()
plt.show()


In [ ]:
if latest_year_drugs:
    top_drugs = pd.concat(latest_year_drugs, ignore_index=True)
    for year, group in top_drugs.groupby("Report year"):
        print(f"\nTop 10 drugs by spending in {year}")
        display(group.sort_values("Spending", ascending=False).head(10).style.format({"Spending": "${:,.0f}"}))


## Additional worksheet samples

In [ ]:
# Each workbook also contains a data dictionary, manufacturer summary, and drug-use information sheet.
for report in reports:
    path, info = report["path"], report["info"]
    print(f"\n## {info['report_year']} workbook")
    for sheet in info["sheets"]:
        if sheet == info["main_sheet"]:
            continue
        sample = pd.read_excel(path, sheet_name=sheet, nrows=8)
        print(f"\n{sheet} — shape in preview: {sample.shape}")
        display(sample.head(5))


## Interpretation and caveats

- The annual reports contain rolling multi-year blocks; this notebook compares the final calendar-year block in each workbook to produce one value for each year from 2016 through 2024.
- The latest-year total is the sum of the reported drug-level spending values. Missing or suppressed source cells remain missing and are not imputed.
- Drug names and row coverage may change across years. Interpret changes in aggregate spending alongside the coverage table and each workbook's methodology.
- These descriptive findings do not establish causes or forecast shortages.